In [1]:
# ============================================================================
# CELL 1 — Install NumPy 2.2.6 + SciPy 1.14.1 into FIX_DIR
# Safe to run repeatedly; papermill-safe; silent on success.
# ============================================================================
import subprocess, sys, os, shutil

FIX_DIR   = "/kaggle/working/numpy_fix"
NUMPY_VER = "2.2.6"
SCIPY_VER = "1.14.1"

def _pair_ok():
    if not os.path.isdir(FIX_DIR):
        return False
    r = subprocess.run(
        [sys.executable, "-c",
         f"import sys; sys.path.insert(0,'{FIX_DIR}');"
         "import numpy, scipy; from scipy.stats import gaussian_kde;"
         f"print(numpy.__version__=='{NUMPY_VER}' and scipy.__version__=='{SCIPY_VER}')"],
        capture_output=True, text=True, timeout=40,
    )
    return r.returncode == 0 and r.stdout.strip() == "True"

if not _pair_ok():
    if os.path.exists(FIX_DIR):
        shutil.rmtree(FIX_DIR)
    print(f"Installing NumPy {NUMPY_VER} + SciPy {SCIPY_VER} into {FIX_DIR} …")
    r = subprocess.run([
        sys.executable, "-m", "pip", "install",
        "--quiet", "--no-warn-script-location", "--disable-pip-version-check",
        "--no-deps", "--root-user-action=ignore", "--target", FIX_DIR,
        f"numpy=={NUMPY_VER}", f"scipy=={SCIPY_VER}",
    ], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr); raise RuntimeError("pip install failed")
    assert _pair_ok(), "numpy+scipy pair verification failed after install"

print(f"✅ NumPy {NUMPY_VER} + SciPy {SCIPY_VER} ready in {FIX_DIR}")

Installing NumPy 2.2.6 + SciPy 1.14.1 into /kaggle/working/numpy_fix …
✅ NumPy 2.2.6 + SciPy 1.14.1 ready in /kaggle/working/numpy_fix


In [2]:
# ============================================================================
# CELL 2 — Adaptive Mobile Money Fraud Detection (papermill-safe, self-healing)
# ============================================================================
import subprocess, sys, os, shutil, importlib

FIX_DIR   = "/kaggle/working/numpy_fix"
NUMPY_VER = "2.2.6"
SCIPY_VER = "1.14.1"

# ---------------------------------------------------------------------------
# STEP 1 — Purge any stale pre-loaded modules
# ---------------------------------------------------------------------------
PURGE_ROOTS = {"numpy", "scipy", "pandas", "matplotlib", "seaborn",
               "sklearn", "lightgbm", "imblearn", "joblib", "river",
               "threadpoolctl"}
preloaded = sorted({m.split(".")[0] for m in sys.modules
                    if m.split(".")[0] in PURGE_ROOTS})
if preloaded:
    print(f"🧹 Purging pre-loaded modules: {preloaded}")
    for m in list(sys.modules):
        if m.split(".")[0] in PURGE_ROOTS:
            del sys.modules[m]

# ---------------------------------------------------------------------------
# STEP 2 — Verify NumPy + SciPy pair in FIX_DIR
# ---------------------------------------------------------------------------
def _run_check(code, expected="True"):
    r = subprocess.run([sys.executable, "-c", code],
                       capture_output=True, text=True, timeout=60)
    return r.returncode == 0 and r.stdout.strip() == expected

def _pair_ok():
    return _run_check(
        f"import sys; sys.path.insert(0,'{FIX_DIR}');"
        "import numpy, scipy; from scipy.stats import gaussian_kde;"
        f"print(numpy.__version__=='{NUMPY_VER}' and scipy.__version__=='{SCIPY_VER}')"
    )

if not _pair_ok():
    if os.path.exists(FIX_DIR):
        shutil.rmtree(FIX_DIR)
    print(f"Installing NumPy {NUMPY_VER} + SciPy {SCIPY_VER} into {FIX_DIR} …")
    r = subprocess.run([
        sys.executable, "-m", "pip", "install",
        "--quiet", "--no-warn-script-location", "--disable-pip-version-check",
        "--no-deps", "--root-user-action=ignore", "--target", FIX_DIR,
        f"numpy=={NUMPY_VER}", f"scipy=={SCIPY_VER}",
    ], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr); raise RuntimeError("numpy+scipy install failed")
    assert _pair_ok(), "numpy+scipy pair verification failed after install"

# ---------------------------------------------------------------------------
# STEP 3 — Install ALL remaining deps into FIX_DIR (this is what was missing)
# ---------------------------------------------------------------------------
REQUIRED = {
    "pandas":        "pandas>=2.0,<2.3",
    "lightgbm":      "lightgbm>=4.0,<5",
    "river":         "river>=0.22",
    "imblearn":      "imbalanced-learn>=0.12,<0.14",
    "joblib":        "joblib>=1.3",
    "sklearn":       "scikit-learn>=1.5,<1.7",
    "threadpoolctl": "threadpoolctl>=3.0",
}

def _importable(name):
    return _run_check(
        f"import sys; sys.path.insert(0,'{FIX_DIR}'); import {name}; print(True)"
    )

missing = [spec for name, spec in REQUIRED.items() if not _importable(name)]

if missing:
    print(f"Installing missing deps into FIX_DIR: {missing}")
    r = subprocess.run([
        sys.executable, "-m", "pip", "install",
        "--quiet", "--no-warn-script-location", "--disable-pip-version-check",
        "--no-deps", "--root-user-action=ignore", "--target", FIX_DIR,
    ] + missing, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr); raise RuntimeError("dependency install failed")

    still_missing = [n for n in REQUIRED if not _importable(n)]
    if still_missing:
        raise RuntimeError(f"Deps still missing after install: {still_missing}")
    print(f"✅ Installed: {', '.join(missing)}")
else:
    print("✅ All dependencies already present")

# ---------------------------------------------------------------------------
# STEP 4 — FIX_DIR to the front of sys.path
# ---------------------------------------------------------------------------
while FIX_DIR in sys.path:
    sys.path.remove(FIX_DIR)
sys.path.insert(0, FIX_DIR)
importlib.invalidate_caches()

# ---------------------------------------------------------------------------
# STEP 5 — Imports (now guaranteed to work)
# ---------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import scipy
import pandas as pd
import matplotlib
matplotlib.use("Agg")                     # headless-safe for papermill
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

import lightgbm as lgb
from sklearn.metrics import (
    classification_report, roc_auc_score, fbeta_score,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, precision_score, recall_score, f1_score,
)
from river.drift import ADWIN
from imblearn.combine import SMOTEENN
import joblib
import river as _river
import imblearn as _imblearn

assert FIX_DIR in np.__file__, f"NumPy loaded from {np.__file__}"
assert FIX_DIR in scipy.__file__, f"SciPy loaded from {scipy.__file__}"
print(f"✅ NumPy {np.__version__} | SciPy {scipy.__version__} | "
      f"lightgbm {lgb.__version__} | river {_river.__version__} | "
      f"imblearn {_imblearn.__version__}")

# ============================================================================
# Global config
# ============================================================================
RANDOM_STATE         = 42
MIN_RECALL_THRESHOLD = 0.85
USE_SMOTEENN         = True
SMOTE_MAJORITY_RATIO = 20
SMOTE_MAX_MAJORITY   = 200_000
OUTPUT_DIR           = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# ============================================================================
# 1. Load data
# ============================================================================
import glob
candidates = [
    "/kaggle/input/datasets/moonknightmarvel/paysim/",
    "/kaggle/input/paysim/", "/kaggle/input/paysim1/",
    "../input/datasets/moonknightmarvel/paysim/", "../input/",
]
csv_files, DATA_PATH = [], None
for c in candidates:
    csv_files = glob.glob(c + "*.csv")
    if csv_files: DATA_PATH = c; break
if not csv_files:
    csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True) or \
                glob.glob("../input/**/*.csv", recursive=True)
    if csv_files: DATA_PATH = csv_files[0].rsplit("/", 1)[0] + "/"
    else: raise FileNotFoundError("PaySim CSV not found — attach the dataset.")

print(f"📁 Using: {csv_files[0]}")
df = pd.read_csv(csv_files[0])
print(f"Shape: {df.shape} | Fraud rate: {df['isFraud'].mean():.4%}")

# ============================================================================
# 2. Preprocessing & feature engineering
# ============================================================================
df["timestamp"] = pd.to_datetime("2020-01-01") + pd.to_timedelta(df["step"], unit="h")
df = pd.concat([df, pd.get_dummies(df["type"], prefix="type")], axis=1)
df["hour"]        = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["is_night"]    = ((df["hour"] >= 22) | (df["hour"] < 6)).astype(int)
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

for col in ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]:
    df[f"{col}_log"] = np.log1p(df[col])

df["error_orig"]       = df["newbalanceOrig"] - (df["oldbalanceOrg"] - df["amount"])
df["error_dest"]       = df["newbalanceDest"] - (df["oldbalanceDest"] + df["amount"])
df["error_orig_abs"]   = np.abs(df["error_orig"])
df["error_dest_abs"]   = np.abs(df["error_dest"])
df["orig_zero_before"] = (df["oldbalanceOrg"]  == 0).astype(int)
df["dest_zero_before"] = (df["oldbalanceDest"] == 0).astype(int)

df = df.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

for role, col in [("orig", "nameOrig"), ("dest", "nameDest")]:
    g = df.groupby(col, sort=False)
    df[f"{role}_tx_count"]   = g.cumcount()
    df[f"{role}_cum_amount"] = g["amount"].cumsum()
    df[f"{role}_avg_amount"] = df[f"{role}_cum_amount"] / (df[f"{role}_tx_count"] + 1)
    df[f"{role}_max_amount"] = g["amount"].cummax()

df["orig_prev_timestamp"]  = df.groupby("nameOrig", sort=False)["timestamp"].shift(1)
df["orig_time_since_last"] = (df["timestamp"] - df["orig_prev_timestamp"]).dt.total_seconds() / 3600
df["orig_time_since_last"] = df["orig_time_since_last"].fillna(0)

def expanding_target_encoding(data, cat_col, target_col="isFraud", alpha=100):
    g_mean      = data[target_col].mean()
    grp         = data.groupby(cat_col, sort=False)[target_col]
    prior_sum   = grp.cumsum() - data[target_col]
    prior_count = grp.cumcount()
    data[f"{cat_col}_te"] = (prior_sum + alpha * g_mean) / (prior_count + alpha)
    return data

df = expanding_target_encoding(df, "nameOrig")
df = expanding_target_encoding(df, "nameDest")

DROP_COLS = ["step", "type", "nameOrig", "nameDest", "isFlaggedFraud",
             "timestamp", "orig_prev_timestamp"]
X_cols = [c for c in df.columns if c not in DROP_COLS + ["isFraud"]]
print(f"Features: {len(X_cols)}")

# ============================================================================
# 3. Helpers
# ============================================================================
def prepare_data(data):
    return data[X_cols].copy().fillna(-999), data["isFraud"].copy()

def train_lightgbm(X_train, y_train, X_val, y_val):
    pos, neg = int(y_train.sum()), int(len(y_train) - y_train.sum())
    spw = max(neg / (pos + 1), 1.0)
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    params = {
        "objective": "binary", "metric": ["auc", "average_precision"],
        "boosting_type": "gbdt", "scale_pos_weight": spw,
        "learning_rate": 0.03, "num_leaves": 63, "max_depth": -1,
        "min_data_in_leaf": 20, "feature_fraction": 0.8,
        "bagging_fraction": 0.8, "bagging_freq": 5,
        "lambda_l1": 0.1, "lambda_l2": 0.1,
        "verbosity": -1, "seed": RANDOM_STATE, "n_jobs": -1,
    }
    return lgb.train(params, dtrain, num_boost_round=1200, valid_sets=[dval],
                     callbacks=[lgb.early_stopping(100, verbose=False),
                                lgb.log_evaluation(200)])

def find_best_threshold(model, X_val, y_val, beta=2, min_recall=MIN_RECALL_THRESHOLD):
    probs = model.predict(X_val)
    prec, rec, thr = precision_recall_curve(y_val, probs)
    prec, rec = prec[:-1], rec[:-1]
    fbeta = (1 + beta**2) * (prec * rec) / (beta**2 * prec + rec + 1e-10)
    valid = rec >= min_recall
    idx = int(np.argmax(np.where(valid, fbeta, -1))) if valid.any() else int(np.argmax(fbeta))
    return float(thr[idx]), float(fbeta[idx])

def evaluate(model, X, y, thr):
    probs = model.predict(X); preds = (probs >= thr).astype(int)
    return {
        "threshold": float(thr), "n_samples": len(y), "n_fraud": int(y.sum()),
        "n_pred_fraud": int(preds.sum()),
        "tp": int(((preds == 1) & (y == 1)).sum()),
        "fp": int(((preds == 1) & (y == 0)).sum()),
        "fn": int(((preds == 0) & (y == 1)).sum()),
        "tn": int(((preds == 0) & (y == 0)).sum()),
        "f2_score":  float(fbeta_score(y, preds, beta=2, zero_division=0)),
        "f1_score":  float(f1_score(y, preds, zero_division=0)),
        "precision": float(precision_score(y, preds, zero_division=0)),
        "recall":    float(recall_score(y, preds, zero_division=0)),
        "auc_roc":   float(roc_auc_score(y, probs)),
        "pr_auc":    float(average_precision_score(y, probs)),
        "mcc":       float(matthews_corrcoef(y, preds)),
    }

def balance_training_data(X, y):
    n_fraud = int(y.sum())
    if (not USE_SMOTEENN) or n_fraud < 50: return X, y
    max_majority = min(SMOTE_MAX_MAJORITY, n_fraud * SMOTE_MAJORITY_RATIO)
    if (len(y) - n_fraud) > max_majority:
        rs = np.random.RandomState(RANDOM_STATE)
        keep = np.concatenate([rs.choice(y[y == 0].index, max_majority, replace=False),
                               y[y == 1].index])
        X, y = X.loc[keep], y.loc[keep]
    try:
        X_res, y_res = SMOTEENN(random_state=RANDOM_STATE).fit_resample(X, y)
        print(f"    SMOTEENN: {len(y):>7,} → {len(y_res):>7,} "
              f"(frauds {int(y.sum()):,} → {int(y_res.sum()):,})")
        return X_res, y_res
    except Exception as e:
        print(f"    SMOTEENN skipped: {e}"); return X, y

# ============================================================================
# 4. Initial training
# ============================================================================
df["date"] = df["timestamp"].dt.date
weeks = sorted(df["date"].unique())
train_end_date = weeks[0] + timedelta(days=7)
df_train = df[df["date"] < train_end_date]
df_rest  = df[df["date"] >= train_end_date]

split_date    = df_train["date"].max() - timedelta(days=1)
df_init_train = df_train[df_train["date"] <  split_date]
df_init_val   = df_train[df_train["date"] >= split_date]
X_tr, y_tr = prepare_data(df_init_train); X_vl, y_vl = prepare_data(df_init_val)

print("\n▶ Initial training")
current_model = train_lightgbm(X_tr, y_tr, X_vl, y_vl)
current_threshold, _ = find_best_threshold(current_model, X_vl, y_vl)
print(f"  Init threshold: {current_threshold:.4f}")
m = evaluate(current_model, X_vl, y_vl, current_threshold)
print(f"  Val → F2 {m['f2_score']:.4f} | Recall {m['recall']:.4f} | "
      f"Precision {m['precision']:.4f} | PR-AUC {m['pr_auc']:.4f}")

# ============================================================================
# 5. Adaptive loop
# ============================================================================
drift_detector  = ADWIN(delta=0.002)
performance_log = []
all_preds, all_probs, all_labels = [], [], []
remaining_weeks = [d for d in weeks if d >= train_end_date]
week_groups = pd.Series(remaining_weeks).groupby(
    pd.to_datetime(remaining_weeks).to_period("W")
).apply(list).tolist()

for week_dates in week_groups:
    df_week = df[df["date"].isin(week_dates)]
    if len(df_week) == 0: continue

    X_wk, y_wk = prepare_data(df_week)
    probs_wk = current_model.predict(X_wk)
    preds_wk = (probs_wk >= current_threshold).astype(int)
    all_probs.extend(probs_wk); all_preds.extend(preds_wk); all_labels.extend(y_wk.values)

    for err in (preds_wk != y_wk.values).astype(int):
        drift_detector.update(int(err))

    wm = evaluate(current_model, X_wk, y_wk, current_threshold)
    wm["week_start"] = str(week_dates[0]); performance_log.append(wm)
    print(f"  Week {week_dates[0]}  n={wm['n_samples']:>7,}  "
          f"F2={wm['f2_score']:.3f}  R={wm['recall']:.3f}  "
          f"P={wm['precision']:.3f}  PR-AUC={wm['pr_auc']:.3f}")

    retrain = False
    if drift_detector.drift_detected:
        print(f"  ⚠️  Drift at {week_dates[0]} — retraining")
        retrain = True; drift_detector.reset()
    if len(performance_log) % 4 == 0:
        print(f"  🔄  Scheduled retrain at {week_dates[0]}")
        retrain = True

    if retrain:
        hist       = df[df["date"] < week_dates[0]]
        hist_dates = sorted(hist["date"].unique())
        cut_date   = hist_dates[int(len(hist_dates) * 0.8)]
        df_rt_tr   = hist[hist["date"] <  cut_date]
        df_rt_vl   = hist[hist["date"] >= cut_date]
        X_rt, y_rt = prepare_data(df_rt_tr); X_rv, y_rv = prepare_data(df_rt_vl)
        print(f"    Retraining on {len(y_rt):,} rows ({int(y_rt.sum()):,} frauds)")
        X_rt_b, y_rt_b = balance_training_data(X_rt, y_rt)
        current_model = train_lightgbm(X_rt_b, y_rt_b, X_rv, y_rv)
        current_threshold, _ = find_best_threshold(current_model, X_rv, y_rv)
        print(f"    New threshold: {current_threshold:.4f}")

# ============================================================================
# 6. Persist artefacts
# ============================================================================
np.save(f"{OUTPUT_DIR}/all_labels.npy", np.array(all_labels))
np.save(f"{OUTPUT_DIR}/all_preds.npy",  np.array(all_preds))
np.save(f"{OUTPUT_DIR}/all_probs.npy",  np.array(all_probs))
pd.DataFrame(performance_log).to_csv(f"{OUTPUT_DIR}/weekly_performance.csv", index=False)
joblib.dump({"model": current_model, "threshold": current_threshold, "X_cols": X_cols},
            f"{OUTPUT_DIR}/adaptive_fraud_model.pkl")
print(f"\n✅ Training done. Artefacts saved to {OUTPUT_DIR}/")

🧹 Purging pre-loaded modules: ['matplotlib', 'numpy']
Installing missing deps into FIX_DIR: ['river>=0.22']
✅ Installed: river>=0.22
✅ NumPy 2.2.6 | SciPy 1.14.1 | lightgbm 4.6.0 | river 0.26.1 | imblearn 0.14.1
📁 Using: /kaggle/input/datasets/moonknightmarvel/paysim/paysim.csv
Shape: (6362620, 11) | Fraud rate: 0.1291%
Features: 36

▶ Initial training
  Init threshold: 1.0000
  Val → F2 0.9953 | Recall 0.9941 | Precision 1.0000 | PR-AUC 0.9998
  Week 2020-01-08  n=2,027,942  F2=0.989  R=0.986  P=1.000  PR-AUC=0.995
  Week 2020-01-13  n=2,065,400  F2=0.985  R=0.985  P=0.986  PR-AUC=0.997
  Week 2020-01-20  n=253,224  F2=0.991  R=0.989  P=1.000  PR-AUC=0.999
  Week 2020-01-27  n= 89,698  F2=0.981  R=0.977  P=1.000  PR-AUC=1.000
  🔄  Scheduled retrain at 2020-01-27
    Retraining on 6,039,437 rows (5,343 frauds)
    SMOTEENN: 112,203 → 208,040 (frauds 5,343 → 105,750)
    New threshold: 0.5458

✅ Training done. Artefacts saved to /kaggle/working/


In [3]:
# ============================================================================
# CELL 3 — Final metrics + pure-Python SVG dashboard (no matplotlib)
# ============================================================================
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    classification_report, fbeta_score, f1_score, precision_score,
    recall_score, roc_auc_score, average_precision_score, matthews_corrcoef,
)

OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

all_labels = np.load(f"{OUTPUT_DIR}/all_labels.npy")
all_preds  = np.load(f"{OUTPUT_DIR}/all_preds.npy")
all_probs  = np.load(f"{OUTPUT_DIR}/all_probs.npy")
perf_df    = pd.read_csv(f"{OUTPUT_DIR}/weekly_performance.csv").sort_values("week_start")

print("\n" + "=" * 64)
print("  FINAL ADAPTIVE MODEL — AGGREGATED TEST PERFORMANCE")
print("=" * 64)
report_str = classification_report(all_labels, all_preds, digits=4,
                                   target_names=["Legit", "Fraud"])
print(report_str)

agg = {
    "F2":        fbeta_score(all_labels, all_preds, beta=2, zero_division=0),
    "F1":        f1_score(all_labels, all_preds, zero_division=0),
    "Precision": precision_score(all_labels, all_preds, zero_division=0),
    "Recall":    recall_score(all_labels, all_preds, zero_division=0),
    "MCC":       matthews_corrcoef(all_labels, all_preds),
    "ROC-AUC":   roc_auc_score(all_labels, all_probs),
    "PR-AUC":    average_precision_score(all_labels, all_probs),
}
for k, v in agg.items():
    print(f"  {k:<10}: {v:.4f}")

pd.DataFrame([agg]).to_csv(f"{OUTPUT_DIR}/final_metrics.csv", index=False)
with open(f"{OUTPUT_DIR}/final_metrics.txt", "w") as fh:
    fh.write(report_str + "\n")
    for k, v in agg.items():
        fh.write(f"{k:<10}: {v:.4f}\n")

# ---- Pure-Python SVG chart generators (same as before) ---------------------
def svg_line(series, title, path, w=760, h=300):
    all_v = [v for vs in series.values() for v in vs]
    lo, hi = min(all_v), max(all_v); rng = (hi - lo) or 1.0
    n = len(next(iter(series.values())))
    ml, mr, mt, mb = 60, 20, 30, 40
    pw, ph = w - ml - mr, h - mt - mb
    o = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}">',
         f'<rect width="{w}" height="{h}" fill="white"/>',
         f'<text x="{w//2}" y="20" text-anchor="middle" font-family="sans-serif" '
         f'font-size="14" font-weight="bold">{title}</text>',
         f'<line x1="{ml}" y1="{mt}" x2="{ml}" y2="{mt+ph}" stroke="#333"/>',
         f'<line x1="{ml}" y1="{mt+ph}" x2="{ml+pw}" y2="{mt+ph}" stroke="#333"/>']
    for i in range(5):
        yy = mt + ph - ph * i / 4; yv = lo + rng * i / 4
        o.append(f'<line x1="{ml-4}" y1="{yy}" x2="{ml}" y2="{yy}" stroke="#333"/>')
        o.append(f'<text x="{ml-8}" y="{yy+4}" text-anchor="end" font-family="sans-serif" '
                 f'font-size="10">{yv:.3f}</text>')
    colors = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]
    for idx, (name, vals) in enumerate(series.items()):
        c = colors[idx % len(colors)]
        pts = " ".join(f"{ml + pw*i/max(1,n-1):.1f},{mt + ph - ph*(v-lo)/rng:.1f}"
                       for i, v in enumerate(vals))
        o.append(f'<polyline points="{pts}" fill="none" stroke="{c}" stroke-width="2"/>')
        lx = ml + 8 + idx * 140
        o.append(f'<line x1="{lx}" y1="{mt-8}" x2="{lx+18}" y2="{mt-8}" '
                 f'stroke="{c}" stroke-width="2"/>')
        o.append(f'<text x="{lx+22}" y="{mt-3}" font-family="sans-serif" font-size="11">{name}</text>')
    o.append("</svg>")
    with open(path, "w") as f: f.write("\n".join(o))

def svg_bar(vals, labels, title, path, color="#1f77b4", w=760, h=340, rotate=0):
    n = len(vals); ymax = max(vals) or 1
    ml, mr, mt, mb = 70, 20, 30, (70 if rotate else 40)
    pw, ph = w - ml - mr, h - mt - mb; bw = pw / n * 0.7
    o = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}">',
         f'<rect width="{w}" height="{h}" fill="white"/>',
         f'<text x="{w//2}" y="20" text-anchor="middle" font-family="sans-serif" '
         f'font-size="14" font-weight="bold">{title}</text>',
         f'<line x1="{ml}" y1="{mt}" x2="{ml}" y2="{mt+ph}" stroke="#333"/>',
         f'<line x1="{ml}" y1="{mt+ph}" x2="{ml+pw}" y2="{mt+ph}" stroke="#333"/>']
    for i in range(5):
        yy = mt + ph - ph * i / 4; yv = ymax * i / 4
        o.append(f'<line x1="{ml-4}" y1="{yy}" x2="{ml}" y2="{yy}" stroke="#333"/>')
        o.append(f'<text x="{ml-6}" y="{yy+4}" text-anchor="end" '
                 f'font-family="sans-serif" font-size="10">{yv:,.0f}</text>')
    step = max(1, n // 12)
    for i, (v, lbl) in enumerate(zip(vals, labels)):
        x = ml + pw * i / n + (pw/n - bw) / 2
        bh = ph * v / ymax; y = mt + ph - bh
        o.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bw:.1f}" '
                 f'height="{bh:.1f}" fill="{color}"/>')
        if i % step == 0:
            cx, cy = x + bw/2, mt + ph + 12
            o.append(f'<text x="{cx:.1f}" y="{cy}" text-anchor="end" '
                     f'font-family="sans-serif" font-size="9" '
                     f'transform="rotate(-45 {cx:.1f} {cy})">{str(lbl)[:10]}</text>')
    o.append("</svg>")
    with open(path, "w") as f: f.write("\n".join(o))

w = perf_df; wk = [str(d) for d in w["week_start"]]
svg_line({"F2": w["f2_score"].tolist()}, "Weekly F2 Score", f"{OUTPUT_DIR}/chart_f2.svg")
svg_line({"Recall": w["recall"].tolist(), "Precision": w["precision"].tolist()},
         "Precision vs Recall", f"{OUTPUT_DIR}/chart_prec_rec.svg")
svg_line({"PR-AUC": w["pr_auc"].tolist(), "ROC-AUC": w["auc_roc"].tolist()},
         "PR-AUC vs ROC-AUC", f"{OUTPUT_DIR}/chart_auc.svg")
svg_line({"MCC": w["mcc"].tolist()}, "Weekly MCC", f"{OUTPUT_DIR}/chart_mcc.svg")
svg_bar(w["n_samples"].tolist(), wk, "Weekly Volume", f"{OUTPUT_DIR}/chart_volume.svg")
svg_bar(w["n_fraud"].tolist(), wk, "Weekly Frauds", f"{OUTPUT_DIR}/chart_frauds.svg",
        color="#d62728")

saved = joblib.load(f"{OUTPUT_DIR}/adaptive_fraud_model.pkl")
booster = saved["model"]
imp_gain = booster.feature_importance(importance_type="gain")
names = booster.feature_name()
pairs = sorted(zip(names, imp_gain), key=lambda x: -x[1])[:20]
svg_bar([p[1] for p in pairs], [p[0] for p in pairs],
        "Top-20 Features (gain)", f"{OUTPUT_DIR}/chart_importance.svg",
        color="#2ca02c", h=520, rotate=1)

print("\n✅ SVG charts generated")

# ---- Bundle into one HTML dashboard ---------------------------------------
charts = ["f2", "prec_rec", "auc", "mcc", "volume", "frauds", "importance"]
html = [
    "<!DOCTYPE html><html><head><meta charset='utf-8'>",
    "<title>Adaptive Fraud Model — Metrics</title><style>",
    "body { font-family: -apple-system, Segoe UI, sans-serif; max-width: 900px;",
    "       margin: 20px auto; padding: 0 20px; color: #222; }",
    "h1 { border-bottom: 3px solid #1f77b4; padding-bottom: 8px; }",
    ".kpi { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; margin: 20px 0; }",
    ".kpi div { background: #f4f8fc; padding: 12px; border-radius: 6px; text-align: center; }",
    ".kpi .v { font-size: 22px; font-weight: bold; color: #1f77b4; }",
    ".kpi .l { font-size: 11px; text-transform: uppercase; color: #666; letter-spacing: 1px; }",
    "svg { display: block; margin: 20px auto; max-width: 100%; height: auto;",
    "      border: 1px solid #eee; border-radius: 6px; padding: 6px; }",
    "</style></head><body>",
    "<h1>Adaptive Mobile Money Fraud Detection</h1>",
    "<p><em>Dataset: PaySim · Aggregated test window · 2026</em></p><div class='kpi'>",
]
for k in ["F2", "Precision", "Recall", "MCC", "F1", "ROC-AUC", "PR-AUC"]:
    html.append(f"<div><div class='v'>{agg[k]:.4f}</div><div class='l'>{k}</div></div>")
html.append("</div>")
for name in charts:
    with open(f"{OUTPUT_DIR}/chart_{name}.svg") as f:
        html.append(f.read())
html.append("</body></html>")
with open(f"{OUTPUT_DIR}/dashboard.html", "w") as f:
    f.write("\n".join(html))
print(f"✅ dashboard.html written")

print(f"\n{'=' * 60}")
print(f"  ✅ PIPELINE COMPLETE")
print(f"{'=' * 60}")
for k in ["F2", "Precision", "Recall", "F1", "MCC", "ROC-AUC", "PR-AUC"]:
    print(f"  {k:<10}: {agg[k]:.4f}")


  FINAL ADAPTIVE MODEL — AGGREGATED TEST PERFORMANCE
              precision    recall  f1-score   support

       Legit     1.0000    1.0000    1.0000   4429953
       Fraud     0.9960    0.9848    0.9904      6311

    accuracy                         1.0000   4436264
   macro avg     0.9980    0.9924    0.9952   4436264
weighted avg     1.0000    1.0000    1.0000   4436264

  F2        : 0.9870
  F1        : 0.9904
  Precision : 0.9960
  Recall    : 0.9848
  MCC       : 0.9904
  ROC-AUC   : 0.9997
  PR-AUC    : 0.9976

✅ SVG charts generated
✅ dashboard.html written

  ✅ PIPELINE COMPLETE
  F2        : 0.9870
  Precision : 0.9960
  Recall    : 0.9848
  F1        : 0.9904
  MCC       : 0.9904
  ROC-AUC   : 0.9997
  PR-AUC    : 0.9976
